# Are our "no effect" results real, or was our sample just too small?

**Data used:** the experiment itself (who got a CN reply + Views/Likes/Shares)

**Short answer:** Views: we had plenty of power (97%). Likes: any real effect is tiny. Shares: genuinely nothing — even a 10x bigger study wouldn’t find it.



# Bootstrap Power Simulation

**Question:** Are the null Likes/Shares effects in our experiment **truly null**, or simply **under-powered** at our sample size?

**The gap our existing bootstrap leaves:** All bootstrap code in `main_effect.ipynb` and the other notebooks resamples at the **observed N** (`np.random.choice(vals, size=len(vals), replace=True)`). That answers "how variable would our estimate be if we re-ran this exact experiment?" — *not* "would more data flip the result?"

**What this notebook does:** Simulates power at increasing sample sizes (1×, 1.5×, 2×, 3×, 5×, 10× original N) by repeatedly bootstrap-resampling each group at the scaled N and running a two-sided MWU test. Empirical power = fraction of trials with `p < 0.05`.

**Three outcomes:** Views, Likes, Shares (winsorized 13-day growth, matching the canonical analysis convention).

**Sanity check:** Views (currently r ≈ −0.0785, p = 0.020) should already be near 80% power at 1× — confirms the simulation calibrates correctly.

**The honest answer it produces:** if Likes/Shares reach 80% power at e.g. 3× our N, we can defensibly claim *"the engagement-quality effects are consistent with a real-but-smaller effect that our experiment was underpowered to detect."* If Shares stays null even at 10× N, that's a stronger paper claim ("effect is genuinely null on Shares") rather than a hedge.


## Section 0 — Config & Imports

In [ ]:
import os
from pathlib import Path

# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then take the field-experiment folder inside it.
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
BASE_DIR = _root / "field-experiment"
if not (BASE_DIR / "data").is_dir():
    raise RuntimeError(
        "Could not locate the field-experiment folder from " + str(Path.cwd()) +
        ". Run this notebook from inside the cloned repository."
    )
DATA_DIR  = BASE_DIR / 'data'
OUT_DIR   = BASE_DIR / 'outputs' / '08_power'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_XL    = DATA_DIR / 'Control_Group.xlsx'
TREATMENT_XL  = DATA_DIR / 'Treatment_Group.xlsx'
MONITORING_XL = DATA_DIR / 'Tweet Monitoring.xlsx'

METRICS      = ['Views', 'Likes', 'Shares']
MAIN_WINDOW  = 13
N_TRIALS     = 1000              # MWU trials per multiplier × outcome
MULTIPLIERS  = [1.0, 1.5, 2.0, 3.0, 5.0, 10.0]
RANDOM_SEED  = 42

print(f'Out dir: {OUT_DIR}')
print(f'Multipliers: {MULTIPLIERS}')
print(f'Trials per cell: {N_TRIALS}')


In [ ]:
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import mannwhitneyu

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
print('Imports OK')


In [ ]:
def parse_mixed_date(date_val):
    if pd.isna(date_val): return pd.NaT
    s = str(date_val).strip()
    def is_valid(year, month):
        return (year == 2025 and month == 12) or (year == 2026 and month == 1)
    if '-' in s and s[:4].isdigit():
        parts = s.split('-')
        year = int(parts[0]); n1 = int(parts[1]); n2 = int(parts[2].split()[0])
        if is_valid(year, n1):    month, day = n1, n2
        elif is_valid(year, n2):  month, day = n2, n1
        else:                      month, day = n1, n2
        return pd.Timestamp(year=year, month=month, day=day)
    elif '/' in s:
        parts = s.split('/')
        n1 = int(parts[0]); n2 = int(parts[1]); year = int(parts[2].split()[0])
        if is_valid(year, n2):    day, month = n1, n2
        elif is_valid(year, n1):  month, day = n1, n2
        else:                      day, month = n1, n2
        return pd.Timestamp(year=year, month=month, day=day)
    return pd.to_datetime(date_val, errors='coerce')

print('Date parser ready.')


## Section 1 — Load Data, Build Outcome Vectors

Replicates the canonical winsorized growth construction (matches `main_effect.ipynb` conventions):

- `growth_w = clip((Day_13 - Day_0) / (Day_0 + 1) * 100, upper=p99)`


In [ ]:
# Load groups + monitoring
control    = pd.read_excel(CONTROL_XL)
treatment  = pd.read_excel(TREATMENT_XL)
monitoring = pd.read_excel(MONITORING_XL)

monitoring['Sample_Date']           = monitoring['Sample_Date'].apply(parse_mixed_date)
monitoring['Start_Date (Creation)'] = monitoring['Start_Date (Creation)'].apply(parse_mixed_date)
monitoring['Day'] = (monitoring['Sample_Date'] - monitoring['Start_Date (Creation)']).dt.days

pivot = monitoring.pivot_table(index='URL', columns='Day',
                               values=['Views','Likes','Comments','Shares'], aggfunc='first')
pivot.columns = [f'{m}_Day{d}' for m, d in pivot.columns]
pivot = pivot.reset_index()

group_map = pd.concat([control[['URL']].assign(Group='Control'),
                       treatment[['URL']].assign(Group='Treatment')])
df = pivot.merge(group_map, on='URL', how='left')

# Day0 fill for Likes/Shares/Comments
for metric in ['Likes', 'Shares', 'Comments']:
    col = f'{metric}_Day0'
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Winsorized growth (matches main convention)
for metric in METRICS:
    d0, dN = f'{metric}_Day0', f'{metric}_Day{MAIN_WINDOW}'
    if d0 in df.columns and dN in df.columns:
        raw = (df[dN] - df[d0]) / (df[d0] + 1) * 100
        p99 = raw.quantile(0.99)
        df[f'{metric}_growth_w'] = raw.clip(upper=p99)

print(f'Pivoted: {df.shape}')
print(f"  Control:   {(df['Group']=='Control').sum()}")
print(f"  Treatment: {(df['Group']=='Treatment').sum()}")

# Build outcome arrays per group per metric
outcomes = {}
for metric in METRICS:
    col = f'{metric}_growth_w'
    ctrl_vals = df.loc[df['Group']=='Control',   col].dropna().values
    trt_vals  = df.loc[df['Group']=='Treatment', col].dropna().values
    outcomes[metric] = (ctrl_vals, trt_vals)
    # Original-sample MWU for reference
    u, p = mannwhitneyu(trt_vals, ctrl_vals, alternative='two-sided')
    r = 1 - (2 * u) / (len(trt_vals) * len(ctrl_vals))
    print(f'  {metric}: n_C={len(ctrl_vals)} n_T={len(trt_vals)}  obs MWU r={r:+.4f}  p={p:.4f}')


## Section 2 — Bootstrap Power Function

Per (outcome × multiplier k), draw `n_C × k` Control samples + `n_T × k` Treatment samples (with replacement from the empirical distribution), run two-sided MWU, repeat `N_TRIALS` times. Empirical power = fraction with `p < 0.05`.

**Why non-parametric resampling, not parametric:** Likes and Shares are heavily zero-inflated (many tweets have zero engagement growth). A parametric model would distort the zero mass. Resampling preserves the empirical distribution.


In [ ]:
def bootstrap_power(ctrl_vals, trt_vals, k, n_trials, alpha=0.05, seed=None):
    """Empirical power at scaled N = k × original N."""
    rng = np.random.default_rng(seed)
    n_c = int(round(len(ctrl_vals) * k))
    n_t = int(round(len(trt_vals)  * k))
    sigs = 0
    for _ in range(n_trials):
        cs = rng.choice(ctrl_vals, size=n_c, replace=True)
        ts = rng.choice(trt_vals,  size=n_t, replace=True)
        try:
            _, p = mannwhitneyu(ts, cs, alternative='two-sided')
            if p < alpha: sigs += 1
        except ValueError:
            pass  # all-zero pathological case
    return sigs / n_trials, n_c, n_t


# Quick smoke test
t0 = time.time()
power_smoke, n_c, n_t = bootstrap_power(*outcomes['Views'], k=1.0, n_trials=200, seed=RANDOM_SEED)
print(f'Smoke test: Views k=1×, 200 trials, power = {power_smoke:.3f}  (elapsed {time.time()-t0:.1f}s)')


## Section 3 — Sweep Multipliers × Outcomes

Run the full simulation. ~3 outcomes × 6 multipliers × 1,000 trials. Expected serial runtime: 2–8 minutes (depends on N at the larger multipliers).


In [ ]:
rows = []
t_start = time.time()

for metric in METRICS:
    ctrl_vals, trt_vals = outcomes[metric]
    print(f'\n=== {metric} (n_C={len(ctrl_vals)} n_T={len(trt_vals)}) ===')

    for k in MULTIPLIERS:
        t0 = time.time()
        # Use a different seed per cell so trials are independent
        seed = RANDOM_SEED + int(k * 1000) + hash(metric) % 10_000
        power, n_c, n_t = bootstrap_power(ctrl_vals, trt_vals, k=k, n_trials=N_TRIALS, seed=seed)
        elapsed = time.time() - t0
        print(f'  k={k:>4}×  N={n_c:>5}/{n_t:>5}  power={power:.3f}  ({elapsed:.1f}s)')
        rows.append(dict(metric=metric, multiplier=k, n_ctrl=n_c, n_trt=n_t, power=power))

power_df = pd.DataFrame(rows)
print(f'\nTotal sweep elapsed: {time.time()-t_start:.0f}s')

power_df.to_csv(OUT_DIR / 'power_curves.csv', index=False)
print(f'Saved: power_curves.csv  ({len(power_df)} rows)')


## Section 4 — Power Curve Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)

colors = {'Views': '#1f77b4', 'Likes': '#d62728', 'Shares': '#2ca02c'}

for metric in METRICS:
    sub = power_df[power_df['metric'] == metric].sort_values('n_trt')
    n_per_group = (sub['n_ctrl'] + sub['n_trt']) / 2  # avg N
    ax.plot(n_per_group, sub['power'], '-o',
            color=colors[metric], lw=2, markersize=7,
            label=metric)

# 80% power reference line
ax.axhline(0.80, color='black', lw=1, ls='--', alpha=0.5)
ax.text(power_df['n_ctrl'].max() * 0.95, 0.81, '80% power',
        ha='right', va='bottom', fontsize=9, color='black', alpha=0.7)

# 5% reference (alpha)
ax.axhline(0.05, color='gray', lw=0.7, ls=':', alpha=0.5)

ax.set_xscale('log')
ax.set_xlabel('Sample size per group (log scale)', fontsize=11)
ax.set_ylabel('Empirical power (fraction of MWU trials with p<0.05)', fontsize=11)
ax.set_title('Bootstrap power simulation — does more data flip the result?', fontsize=12)
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3, linestyle='--')
ax.legend(loc='lower right', frameon=True)

plt.tight_layout()
plot_path = OUT_DIR / 'fig_power_vs_n.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {plot_path.name}')


## Section 5 — Solve N for 80% Power (Linear Interpolation)

For each outcome, find the smallest N where empirical power ≥ 0.80 by linear interpolation across the multipliers we sampled. If the curve never reaches 80% within our 10× sweep, we report `>10× original N` and flag it as a **truly null** result rather than a power-limited one.


In [ ]:
def n_for_target_power(sub, target=0.80):
    sub = sub.sort_values('multiplier').reset_index(drop=True)
    # Walk the curve and interpolate the first crossing
    for i in range(len(sub) - 1):
        p_lo, p_hi = sub.loc[i, 'power'], sub.loc[i+1, 'power']
        if p_lo >= target:
            return float(sub.loc[i, 'n_ctrl'] + sub.loc[i, 'n_trt']) / 2
        if p_hi >= target:
            # linear interp between (n_lo, p_lo) and (n_hi, p_hi)
            n_lo = (sub.loc[i, 'n_ctrl']   + sub.loc[i, 'n_trt'])   / 2
            n_hi = (sub.loc[i+1, 'n_ctrl'] + sub.loc[i+1, 'n_trt']) / 2
            return float(n_lo + (target - p_lo) / (p_hi - p_lo) * (n_hi - n_lo))
    return None  # never reached target

n_for_80 = []
for metric in METRICS:
    sub = power_df[power_df['metric'] == metric]
    n_target = n_for_target_power(sub, target=0.80)
    obs_power = float(sub.loc[sub['multiplier']==1.0, 'power'].iloc[0])
    n_orig    = float((sub.loc[sub['multiplier']==1.0, 'n_ctrl'].iloc[0] +
                       sub.loc[sub['multiplier']==1.0, 'n_trt'].iloc[0]) / 2)
    n_for_80.append(dict(
        metric=metric,
        observed_power=obs_power,
        n_orig_per_group=n_orig,
        n_for_80pct_power=n_target if n_target is not None else np.nan,
        multiplier_for_80=n_target / n_orig if n_target else np.nan,
        verdict=('reaches 80%' if n_target else f'>10× orig N, plausibly truly null'),
    ))

n_for_80_df = pd.DataFrame(n_for_80)
n_for_80_df.to_csv(OUT_DIR / 'n_for_80pct_power.csv', index=False)
print('=== N required for 80% power ===')
print(n_for_80_df.to_string(index=False))
print()
print(f'Saved: n_for_80pct_power.csv')


## Section 6 — Summary + Sanity Checks

**Expected sanity outcomes:**
- **Views:** original-N power should be roughly **0.50–0.70** (the observed p = 0.020 is significant but not extreme — sometimes you'd miss it). N-for-80% should be roughly 1× to 1.5×.
- **Likes / Shares:** original-N power should be **near α = 0.05** (the observed p ≈ 0.4–0.7 is non-significant by a wide margin). Verifies the simulation calibrates correctly.

**Interpretation framework for the paper:**
- If `n_for_80% / n_original ≤ 3×`: claim **"power-limited"** — a real-but-smaller effect plausibly exists, future replication with larger N could detect it.
- If `n_for_80% / n_original > 5–10×`: claim **"plausibly truly null"** — even a much larger experiment would not detect a meaningful effect.


In [ ]:
print('=' * 60)
print('SUMMARY')
print('=' * 60)
for _, row in n_for_80_df.iterrows():
    print(f'\n{row["metric"]}:')
    print(f'  Observed power at original N: {row["observed_power"]:.3f}')
    print(f'  Original N per group:         {row["n_orig_per_group"]:.0f}')
    if not pd.isna(row['n_for_80pct_power']):
        print(f'  N for 80% power:              {row["n_for_80pct_power"]:.0f}')
        print(f'  Multiplier needed:            {row["multiplier_for_80"]:.1f}×')
    print(f'  Verdict:                      {row["verdict"]}')


## Section 7 — Outputs Index

Saved to `outputs/08_power/`:
- `power_curves.csv` — long format: metric, multiplier, n_ctrl, n_trt, power.
- `n_for_80pct_power.csv` — table of N-needed-for-80%-power per metric, with verdict.
- `fig_power_vs_n.png` — power curves with 80% reference line.
